# 03 — Milestone I: full frozen comparison
Run notebook 00 first on the same Colab GPU kernel. This notebook prepares the four locked test sets, trains both frozen backbones on all 4,040 training images for 40 epochs, evaluates every test set independently, and creates quantitative plots plus paired qualitative overlays. Test data is never used for training or checkpoint selection.

In [ ]:
from pathlib import Path
import os, subprocess, sys, torch

PROJECT_DIR = Path('/content/cod-ssl')
DATA_ROOT = Path('/content/drive/MyDrive/cod-ssl/data')
RUNS_ROOT = Path('/content/drive/MyDrive/cod-ssl/runs')
COMPARISONS_ROOT = Path('/content/drive/MyDrive/cod-ssl/comparisons')
ACCEPT_COD10K_NONCOMMERCIAL_LICENSE = True  # Review the SINet license, then set True.
EPOCHS = 40
QUALITATIVE_COUNT = 24
EXPERIMENT_TAG = 'phase1_seed42'  # Stable names make interrupted runs resumable.
assert PROJECT_DIR.is_dir(), 'Run notebook 00 first.'
os.chdir(PROJECT_DIR)

In [ ]:
# Download/cache/extract CAMO-Test, COD10K-Test, CHAMELEON and NC4K; then build manifests.
if not ACCEPT_COD10K_NONCOMMERCIAL_LICENSE:
    raise PermissionError('Review https://github.com/DengPingFan/SINet#9-license, then set the acceptance flag to True.')
subprocess.run([
    sys.executable, 'scripts/bootstrap_test_data.py',
    '--data-root', str(DATA_ROOT), '--manifest-dir', 'manifests',
    '--accept-noncommercial-license',
], check=True)
subprocess.run([sys.executable, 'scripts/validate_dataset.py', '--manifest-dir', 'manifests'], check=True)
os.sync()

In [ ]:
# Stable Drive paths make Colab disconnects recoverable. Completed runs are skipped.
DINO_RUN = RUNS_ROOT / f'{EXPERIMENT_TAG}_dinov3_vitb16'
VJEPA_RUN = RUNS_ROOT / f'{EXPERIMENT_TAG}_vjepa21_vitb16'
COMPARISON_DIR = COMPARISONS_ROOT / f'{EXPERIMENT_TAG}_frozen_comparison'
def train_or_resume(config, run_dir):
    last = run_dir / 'checkpoints' / 'last.pt'
    epoch_checkpoints = sorted((run_dir / 'checkpoints').glob('epoch_*.pt'))
    checkpoint = last if last.is_file() else (epoch_checkpoints[-1] if epoch_checkpoints else None)
    if checkpoint is not None:
        epoch = int(torch.load(checkpoint, map_location='cpu', weights_only=False)['epoch'])
        if epoch >= EPOCHS:
            print(f'Already complete ({epoch} epochs): {run_dir}')
            return
    command = [sys.executable, 'scripts/train.py', '--config', config,
               '--run-dir', str(run_dir), '--epochs', str(EPOCHS)]
    if checkpoint is not None:
        command += ['--resume', str(checkpoint)]
        print('Resuming from', checkpoint)
    subprocess.run(command, check=True)
    os.sync()
print('DINO_RUN =', DINO_RUN)
print('VJEPA_RUN =', VJEPA_RUN)
print('COMPARISON_DIR =', COMPARISON_DIR)

In [ ]:
# Full DINOv3 frozen run: all 4,040 images, locked Phase-1 configuration.
train_or_resume('configs/frozen_dinov3_vitb16.yaml', DINO_RUN)

In [ ]:
# Full V-JEPA 2.1 frozen run with the identical decoder and training protocol.
train_or_resume('configs/frozen_vjepa21_vitb16.yaml', VJEPA_RUN)

In [ ]:
# Evaluate each final checkpoint independently on all four locked test sets.
for run in (DINO_RUN, VJEPA_RUN):
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--run', str(run)], check=True)
os.sync()

In [ ]:
# Build metric/compute tables, comparison graphs, and paired qualitative overlays.
subprocess.run([sys.executable, 'scripts/compare_runs.py',
    str(DINO_RUN), str(VJEPA_RUN), '--output', str(COMPARISON_DIR),
    '--qualitative-count', str(QUALITATIVE_COUNT)], check=True)
os.sync()

In [ ]:
# Display the summary artifacts inline; all originals remain persisted in Drive.
from IPython.display import display
from PIL import Image
import pandas as pd
display(pd.read_csv(COMPARISON_DIR / 'comparison_metrics.csv'))
display(pd.read_csv(COMPARISON_DIR / 'compute_comparison.csv'))
display(Image.open(COMPARISON_DIR / 'metric_comparison.png'))
display(Image.open(COMPARISON_DIR / 'training_curves.png'))
panel_paths = sorted((COMPARISON_DIR / 'qualitative_panels').glob('*.png'))
for path in panel_paths[:8]:
    print(path.name); display(Image.open(path))
print(f'All {len(panel_paths)} panels:', COMPARISON_DIR / 'qualitative_panels')